# 01 — Exploratory data analysis

What this notebook answers, and the decision each answer feeds:

| Question | Feeds |
|---|---|
| How are labels distributed over time? | Context for drift (Phase 5) |
| How many illicit transactions per training step? | K for precision@K (D-004) |
| What do the features look like (scale, skew, constants, duplicates)? | Preprocessing for Phase 2 |
| What does the graph look like per step? | Neo4j/GNN design (Phases 3–4) |
| Do illicit transactions connect to each other (homophily)? | Whether graph features can plausibly help |

**Rule (D-010):** decisions use the training period (steps 1–34) only. Section 1 shows all 49
steps *for context only*
everything after it is computed on steps 1–34.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from fraudgraph.config import resolve
from fraudgraph.data.load import ILLICIT, LICIT, UNKNOWN, load_elliptic
from fraudgraph.data.split import load_split_config

FIG_DIR = resolve("docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

data = load_elliptic()  # validated against the published counts on load
nodes, edges = data.nodes, data.edges
split = load_split_config()
TRAIN_PERIOD_END = split.val_steps[1]  # 34: train + validation = everything before test
print(f"{len(nodes):,} nodes, {len(edges):,} edges, {len(data.feature_cols)} features")

## 1. Labels over time (context only — includes the test period)

Stacked counts of illicit / licit / unknown per step. Dashed lines mark the split boundaries.
This is the one place the test period is shown
nothing here drives a decision.

In [ ]:
counts = (
    nodes.groupby(["time_step", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={ILLICIT: "illicit", LICIT: "licit", UNKNOWN: "unknown"})
)
counts["illicit_share_of_labelled"] = counts["illicit"] / (counts["illicit"] + counts["licit"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])
ax1.bar(counts.index, counts["licit"], label="licit", color="#4c78a8")
ax1.bar(counts.index, counts["illicit"], bottom=counts["licit"], label="illicit", color="#e45756")
ax1.set_ylabel("labelled transactions")
ax1.legend(frameon=False)
ax2.plot(counts.index, counts["illicit_share_of_labelled"], marker="o", ms=3, color="#e45756")
ax2.set_ylabel("illicit share")
ax2.set_xlabel("time step")
for ax in (ax1, ax2):
    for b in (split.val_steps[0] - 0.5, split.test_steps[0] - 0.5):
        ax.axvline(b, ls="--", color="grey", lw=1)
ax1.set_title("Labels per time step (dashed: train | val | test)")
fig.tight_layout()
fig.savefig(FIG_DIR / "labels_per_step.png")
plt.show()
counts

## 2. Illicit transactions per training step → choosing K (D-004)

K stands for an investigator team's review capacity per step. The data's only role is a sanity
check: if K is far above the typical number of illicit transactions in a step, precision@K is
capped below 1 even for a perfect model, and the metric stops measuring the model.

In [ ]:
train_period = nodes[nodes["time_step"] <= TRAIN_PERIOD_END]
illicit_per_step = (train_period["label"] == ILLICIT).groupby(train_period["time_step"]).sum()
print(illicit_per_step.describe().round(1).to_string())
for k in (10, 20, 50, 100):
    capped = (illicit_per_step < k).mean()
    print(
        f"K={k:>3}: {capped:.0%} of training steps have fewer than K illicit "
        f"(precision@K capped below 1 there)"
    )

## 3. Features (training period, labelled + unknown)

Checks that decide preprocessing: are features already scaled? heavily skewed? any constant
or exactly duplicated columns that carry no extra information?

In [ ]:
X = train_period[data.feature_cols]
desc = X.describe().T
print(
    "mean of column means:",
    round(desc["mean"].mean(), 3),
    "| mean of column stds:",
    round(desc["std"].mean(), 3),
)
skew = X.skew()
print(f"columns with |skew| > 5: {(skew.abs() > 5).sum()} of {len(skew)}")
const = desc.index[desc["std"] == 0].tolist()
print("constant columns:", const)

# Highly correlated pairs (|r| > 0.99): near-duplicates. Fine for trees, but they make
# logistic-regression coefficients unstable and inflate apparent feature counts.
corr = X.corr().abs().to_numpy()
iu = np.triu_indices_from(corr, k=1)
cols = data.feature_cols
pairs = [(cols[i], cols[j]) for i, j in zip(*iu, strict=True) if corr[i, j] > 0.99]
print(f"pairs with |r| > 0.99: {len(pairs)}")
print(pairs[:10])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, ["local_1", "local_2", "agg_1"], strict=True):
    ax.hist(X[col], bins=80, color="#4c78a8")
    ax.set_yscale("log")
    ax.set_title(col)
fig.suptitle("Example feature distributions (log count)")
fig.tight_layout()
plt.show()

## 4. Graph structure per step (training period)

Kaggle's description says each step is a single connected component. We check it, and look at
degree: how many flows each transaction takes part in.

In [ ]:
tp_ids = set(train_period["tx_id"])
tp_edges = edges[edges["src"].isin(tp_ids)]  # edges never cross steps, so src decides
step_of = nodes.set_index("tx_id")["time_step"]

rows = []
for step, g in tp_edges.groupby(tp_edges["src"].map(step_of)):
    ids = nodes.loc[nodes["time_step"] == step, "tx_id"].to_numpy()
    idx = pd.Series(np.arange(len(ids)), index=ids)
    a = coo_matrix(
        (np.ones(len(g)), (idx[g["src"]].to_numpy(), idx[g["dst"]].to_numpy())),
        shape=(len(ids), len(ids)),
    )
    n_comp, comp = connected_components(a, directed=False)
    rows.append(
        {
            "step": step,
            "nodes": len(ids),
            "edges": len(g),
            "components": n_comp,
            "largest_component_share": np.bincount(comp).max() / len(ids),
        }
    )
graph_stats = pd.DataFrame(rows).set_index("step")
graph_stats.describe().round(3)

In [ ]:
deg = pd.concat([tp_edges["src"], tp_edges["dst"]]).value_counts()
deg = deg.reindex(train_period["tx_id"], fill_value=0)
print(deg.describe().round(2).to_string())
vc = deg.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(vc.index[vc.index > 0], vc[vc.index > 0], "o", ms=3, color="#4c78a8")
ax.set_xlabel("degree (in + out)")
ax.set_ylabel("number of transactions")
ax.set_title("Degree distribution (training period)")
fig.tight_layout()
plt.show()

## 5. Homophily: do illicit transactions link to illicit transactions?

If labels were unrelated to the graph, an edge between two labelled transactions would be
illicit–illicit about as often as random pairing predicts (`p_illicit²`). Much more often than
that means the graph carries signal, so graph features and GNNs have something to learn from.
Edges touching an unknown node are excluded from this table.

In [ ]:
label_of = nodes.set_index("tx_id")["label"]
el = tp_edges.assign(ls=tp_edges["src"].map(label_of), ld=tp_edges["dst"].map(label_of))
el = el[(el["ls"] != UNKNOWN) & (el["ld"] != UNKNOWN)]
names = {ILLICIT: "illicit", LICIT: "licit"}
pair_counts = pd.crosstab(el["ls"].map(names), el["ld"].map(names))
print("labelled–labelled edges by (src, dst) label:\n", pair_counts, "\n")

lab = train_period[train_period["label"] != UNKNOWN]
p = (lab["label"] == ILLICIT).mean()
obs = ((el["ls"] == ILLICIT) & (el["ld"] == ILLICIT)).mean()
print(f"illicit share among labelled nodes: {p:.3f}")
print(f"illicit–illicit edge share: observed {obs:.4f} vs {p**2:.4f} if random ({obs / p**2:.1f}x)")

In [ ]:
# Per-node view: share of a node's labelled neighbours that are illicit, by the node's own label.
# Exploration only — this is exactly the label-based feature D-001 rules out for modelling.
und = pd.concat(
    [
        el[["src", "dst", "ld"]].rename(columns={"src": "n", "ld": "nl"}),
        el[["dst", "src", "ls"]].rename(columns={"dst": "n", "ls": "nl"}),
    ]
)
nbr_rate = (und["nl"] == ILLICIT).groupby(und["n"]).mean()
own = label_of.reindex(nbr_rate.index).map(names)
print(nbr_rate.groupby(own).describe().round(3).to_string())
print(
    f"\nlabelled training-period nodes with no labelled neighbour: "
    f"{1 - len(nbr_rate) / len(lab):.0%}"
)

## Findings

Written up in the README's *EDA findings* section, so they sit next to the results.